In [ ]:
import os
# prevent OMP conflict between torch and sklearn
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)
os.makedirs("../figures", exist_ok=True)

Upload the labels.csv and processed_counts.csv files to colab or your local workspace.

This data associates a cell barcode, such as "AAAGCCTGGCTAAC-1", to a certain cell type label, such as "CD14+ Monocyte". For each cell barcode, there are also log RNA seq counts of 765 different genes, such as HES4.

label.csv stores the association between a cell barcode and a cell type label.

processed_counts.csv stores the normalized log read counts for each cell, where each row represents a single cell, and each column represents a gene.

In [ ]:
labels_pd = pd.read_csv("../data/labels.csv")
counts_pd = pd.read_csv("../data/processed_counts.csv")

In [ ]:
# Constants and data preprocessing: set indices, merge on barcode, separate features and labels
INPUT_DIM = 765
LATENT_DIM = 32

labels_pd.index = labels_pd['index']
labels_pd.drop("index", axis=1, inplace=True)
counts_pd.index = counts_pd['Unnamed: 0']
counts_pd.drop("Unnamed: 0", axis=1, inplace=True)

df = counts_pd.merge(labels_pd, left_index=True, right_index=True).dropna()
df.shape, df['bulk_labels'].value_counts()

Shuffle your data. Make sure your labels and the counts are shuffled together.

Split into train and test sets (80:20 split)

In [ ]:
le = LabelEncoder()
X = df.drop('bulk_labels', axis=1).values.astype(np.float32)
y = le.fit_transform(df['bulk_labels'].values)
y_labels = df['bulk_labels'].values

np.random.seed(42)
idx = np.random.permutation(len(X))
X, y, y_labels = X[idx], y[idx], y_labels[idx]

X_train, X_test = X[:560], X[560:]
y_train, y_test = y[:560], y[560:]

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Create a fully connected neural network for your autoencoder. Your latent space can be of any size less than or equal to 64. Too large may result in a poor visualization, and too small may result in high loss. 32 is a good starting point.

Consider using more than 1 hidden layer, and a sparcity constraint (l1 regularization).

Have an encoder model which is a model of only the layers for the encoding.

In [ ]:
# Fully connected autoencoder: 765->512->256->128->32->128->256->512->765 with ReLU activations
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(512, 256),       nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128),       nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256),        nn.ReLU(),
            nn.Linear(256, 512),        nn.ReLU(),
            nn.Linear(512, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

    def encode(self, x):
        return self.encoder(x)

In [ ]:
# Standalone encoder matching the first half of the autoencoder
class Encoder(nn.Module):
    def __init__(self, autoencoder):
        super().__init__()
        self.net = autoencoder.encoder

    def forward(self, x):
        return self.net(x)

Train your autoencoding using MSE loss.

Finally, identify the parameters which don't overfit, and use the same model architecture and train on all of the data together.

With a latent space size of 32, aim for 0.9 MSE loss on your test set, 0.95 with regularization. You will not be graded strictly on a loss cutoff.

In [ ]:
# Train autoencoder with MSE + L1 sparsity regularization; early stopping on val loss
def fit(X_tr, X_val, epochs=200, lr=1e-3, l1=1e-4, batch_size=64):
    model = Autoencoder(X_tr.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    mse = nn.MSELoss()

    tr_t  = torch.FloatTensor(X_tr)
    val_t = torch.FloatTensor(X_val)
    loader = DataLoader(TensorDataset(tr_t), batch_size=batch_size, shuffle=True)

    train_hist, val_hist = [], []
    best, best_epoch, wait = float('inf'), 0, 0

    for epoch in range(epochs):
        model.train()
        batch_loss = 0
        for (xb,) in loader:
            opt.zero_grad()
            xhat, z = model(xb)
            loss = mse(xhat, xb) + l1 * z.abs().mean()
            loss.backward()
            opt.step()
            batch_loss += mse(xhat, xb).item()
        train_hist.append(batch_loss / len(loader))

        model.eval()
        with torch.no_grad():
            xhat_val, _ = model(val_t)
            v = mse(xhat_val, val_t).item()
        val_hist.append(v)

        if v < best:
            best, best_epoch, wait = v, epoch, 0
            torch.save(model.state_dict(), '/tmp/ae_best.pt')
        else:
            wait += 1
            if wait == 20:
                break

    model.load_state_dict(torch.load('/tmp/ae_best.pt', weights_only=True))
    return model, train_hist, val_hist


# use 80% of train to fit, 20% to validate hyperparams
X_tr, X_val = X_train[:448], X_train[448:]
model, train_hist, val_hist = fit(X_tr, X_val)

model.eval()
with torch.no_grad():
    xhat_test, _ = model(torch.FloatTensor(X_test))
print(f"test MSE: {nn.MSELoss()(xhat_test, torch.FloatTensor(X_test)):.4f}")

plt.figure(figsize=(7, 4))
plt.plot(train_hist, label='train')
plt.plot(val_hist, label='val')
plt.xlabel('epoch'); plt.ylabel('MSE')
plt.legend(); plt.tight_layout()
plt.savefig('../figures/training_curve.png', dpi=150)
plt.show()

In [ ]:
# Retrain same architecture on all data and extract encoder
all_t = torch.FloatTensor(X)
loader_all = DataLoader(TensorDataset(all_t), batch_size=64, shuffle=True)

final = Autoencoder(X.shape[1])
opt_f = torch.optim.Adam(final.parameters(), lr=1e-3, weight_decay=1e-4)
loss_f = nn.MSELoss()

final.train()
for epoch in range(150):
    for (xb,) in loader_all:
        opt_f.zero_grad()
        xhat, z = final(xb)
        loss = loss_f(xhat, xb) + 1e-4 * z.abs().mean()
        loss.backward()
        opt_f.step()

final.eval()
with torch.no_grad():
    _, Z = final(all_t)
Z = Z.numpy()

encoder = Encoder(final)

Use PCA and t-SNE on the dataset.

Then use PCA on the latent space representation of the dataset.

Plot all of these.

In [ ]:
# PCA on raw gene expression projected to 2D, colored by cell type
# shared colors across all three plots
cell_types = le.classes_
cmap = matplotlib.colormaps.get_cmap('tab10').resampled(len(cell_types))
color_map = {ct: cmap(i) for i, ct in enumerate(cell_types)}
colors = [color_map[ct] for ct in y_labels]
patches = [mpatches.Patch(color=color_map[ct], label=ct) for ct in cell_types]

pca2 = PCA(n_components=2, random_state=42)
X_pca = pca2.fit_transform(X)

In [ ]:
# t-SNE on top of 50 PCs — standard for this data size
X_pc50 = PCA(n_components=50, random_state=42).fit_transform(X)
X_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_pc50)

In [ ]:
# PCA on autoencoder latent representations
Z_pca = PCA(n_components=2, random_state=42).fit_transform(Z)


def scatter(ax, data, title, xl, yl):
    ax.scatter(data[:, 0], data[:, 1], c=colors, s=12, alpha=0.85)
    ax.set_title(title); ax.set_xlabel(xl); ax.set_ylabel(yl)


fig, axes = plt.subplots(1, 3, figsize=(20, 6))
scatter(axes[0], X_pca,  'PCA',                       'PC1',     'PC2')
scatter(axes[1], X_tsne, 't-SNE',                     't-SNE 1', 't-SNE 2')
scatter(axes[2], Z_pca,  'Autoencoder latent (PCA)',  'PC1',     'PC2')
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=8,
           bbox_to_anchor=(0.5, -0.12))
plt.suptitle('PBMC dimensionality reduction', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../figures/part1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
for data, fname, title, xl, yl in [
    (X_pca,  'pca.png',             'PCA',                      'PC1',     'PC2'),
    (X_tsne, 'tsne.png',            't-SNE',                    't-SNE 1', 't-SNE 2'),
    (Z_pca,  'autoencoder_pca.png', 'Autoencoder latent (PCA)', 'PC1',     'PC2'),
]:
    fig, ax = plt.subplots(figsize=(7, 6))
    scatter(ax, data, title, xl, yl)
    ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
    plt.tight_layout()
    plt.savefig(f'../figures/{fname}', dpi=150, bbox_inches='tight')
    plt.show()

Compare the results of PCA, t-SNE, and your autoencoder as ways to visualize the data.

## Comparison

t-SNE gave the clearest separation of the three methods. Rare populations like CD34+ and CD4+/CD45RA+/CD25- Naive T cells are nearly invisible in PCA but show up as distinct clusters in t-SNE. PCA is a linear projection, so T cell subtypes, which differ in subtle ways across many dimensions, tend to collapse onto each other in the first two PCs.

The autoencoder latent space is harder to read visually. Even after projecting the 32 dimensional encoding down with PCA, there is substantial cluster overlap. The model was trained for reconstruction, not class separation, and with 1.1M parameters on 560 training samples it still overfits somewhat despite dropout (train MSE ~0.79, val MSE ~0.87). Dropout brought the gap down from ~0.19 to ~0.09, which helped. For visualization t-SNE is clearly the better choice; the autoencoder encoding is more useful as a feature representation for a downstream classifier.